# 🎬 مولّد فيديوهات الإعلانات — Wan 2.1 النسخة الخفيفة (مجاني + مفتوح المصدر)

**هذه النسخة تحل مشكلة انهيار الذاكرة** اللي صارت بالنسخة السابقة — مبنية على وصفة مجرّبة تشتغل فعلياً على كولاب المجاني T4.

## وش الفرق عن النسخة السابقة؟
| | النسخة السابقة | **هذه النسخة** ✓ |
|---|---|---|
| نوع التحميل | كامل 27GB | **مضغوط GGUF ~16GB** |
| استهلاك الذاكرة | انهار (78GB و13GB) | **مضبوط على T4** |
| النتيجة | تجمد بعد Save | **تعمل** |

## التوقعات الصادقة (على كولاب المجاني):
- كل لقطة: **1.5–2 ثانية** بدقة 480p (افقي أو عمودي)
- **كل توليدة ~20–28 دقيقة** على T4 (هذه طبيعة كولاب المجاني)
- الحركة الاحترافية: ولّد **4–8 لقطات** ووصلها بمونتاج CapCut بتسلسل إعلاني كامل مع الشعار والنص
- خالٍ من العلامة المائية، ورخصته Apache 2.0 → **ملكك واستخدامه بإعلانات ممولة مسموح**

## كيف تشغّله؟
1. **Runtime → Change runtime type → T4 GPU**
2. اضغط `Shift+Enter` على الخانات بالترتيب
3. الخانة الأخيرة فيها **حقول النص والجودة** — عدّل وصف قصتك واضغط تشغيل

> ⚠️ بعد فتح النوت بوك: **Runtime → Restart session** أولاً (لأن الجلسة السابقة انهارت)، ثم Run all.

In [ ]:
# ===== الخطوة 1: تجهيز البيئة (مرة واحدة لكل جلسة) =====
# نسخة خفيفة (GGUF) مجرّبة على كولاب المجاني T4 — بلا انهيار ذاكرة
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content

!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install av
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

# اختيار الجودة: Q5 أسرع (الأساسي) | Q6 أوضح لكن أثقل — كلاهما يعمل على T4
useQ6 = False # @param {"type":"boolean"}

# تحميل النموذج الرئيسي (المضغوط GGUF ~10GB):
if useQ6:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q6_K.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q6_K.gguf
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q5_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q5_0.gguf

# مشفر النصوص المضغوط fp8 (~6GB) — هاذا الجزء كان ينهي الذاكرة بالنسخة الكاملة
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
# ترميز الفيديو VAE:
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
import subprocess
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM

unet_loader = UnetLoaderGGUF()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'lossless': bool(lossless),
        'method': int(method)
    }

    with imageio.get_writer(
        output_path,
        format='WEBP',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'codec': str(codec),
        'output_params': ['-crf', str(int(quality))]
    }

    with imageio.get_writer(
        output_path,
        format='FFMPEG',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"

    frame = (image.cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(frame).save(output_path)
    return output_path

def generate_video(
    positive_prompt: str = "A sleek smartphone on a desk, an elegant Arabic AI chat app interface with messages, golden and deep blue lighting, cinematic slow zoom, premium tech commercial",
    negative_prompt: str = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走",
    width: int = 832,
    height: int = 480,
    seed: int = 82628696717253,
    steps: int = 20,
    cfg_scale: float = 3.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 24,
    fps: int = 16,
    output_format: str = "mp4"
):
    with torch.inference_mode():
        print("جارٍ تحميل مشفر النصوص...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]

        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()

        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]

        print("جارٍ تحميل مولّد الفيديو...")
        if useQ6:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q6_K.gguf")[0]
        else:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q5_0.gguf")[0]

        print("⏳ جاري التوليد... ياخذ 20-28 دقيقة على T4")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()

        print("جارٍ تحويل النتيجة إلى فيديو...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]

        try:
            decoded = vae_decode.decode(vae, sampled)[0]

            del vae
            torch.cuda.empty_cache()
            gc.collect()

            output_path = ""
            if frames == 1:
                print("إطار واحد → صورة PNG")
                output_path = save_as_image(decoded[0], "ComfyUI")
                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    output_path = save_as_webm(decoded, "ComfyUI", fps=fps, codec="vp9", quality=10)
                elif output_format.lower() == "mp4":
                    output_path = save_as_mp4(decoded, "ComfyUI", fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")

                display_video(output_path)

        except Exception as e:
            print(f"خطأ أثناء الحفظ: {str(e)}")
            raise
        finally:
            clear_memory()
        return output_path

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    video_data = open(video_path,'rb').read()

    if video_path.lower().endswith('.mp4'):
        mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'):
        mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'):
        mime_type = "image/webp"
    else:
        mime_type = "video/mp4"

    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))

print("✅ البيئة جاهزة!")

## ✍️ نصوص إعلانية جاهزة (الصقة أحدها في خانة التوليد بالأسفل)

| المقطع | النص بالإنجليزي (نتائج أفضل) |
|---|---|
| **تعريفي** | *A sleek smartphone on a desk, an elegant Arabic AI chat app interface with messages, golden and deep blue lighting, cinematic slow zoom, premium tech commercial* |
| **مقارنة** | *Split screen: stressed person drowning in tasks on the left, confident person using a smart Arabic AI assistant on the right, clean bright advertising style, smooth transition* |
| **إبهار تقني** | *Abstract streams of glowing Arabic calligraphy data forming interface windows in the air, futuristic fintech aesthetic, slow motion golden particles, elegant* |
| **صديق ذكي** | *Person speaks to a glowing AI orb that projects beautiful artwork into the air, dark studio with neon blue and gold accents, cinematic camera orbit* |
| **خصوصية** | *A person chatting with an AI on their own laptop in a cozy room, warm lighting, subtle camera drift, reassuring atmosphere, high-end tech ad* |

**سرّ الحركة الجيدة:** أضف لكلمات الوصف: `cinematic slow zoom`, `camera orbit`, `gentle pan`, `smooth motion`.

**التحذير الأصلي `negative_prompt` مكتوب بالصينية — لا تمسّه** (هو تحذير الملف الرفيع الرسمي يمنع الأيدي المشوهة والجودة السيئة).

In [ ]:
# ===== الخطوة 2: توليد الفيديو =====
# الصقة وصف لقطة إعلانية في positive_prompt، واضبط الجودة ثم شغّل
positive_prompt = "A sleek smartphone on a desk, an elegant Arabic AI chat app interface with messages, golden and deep blue lighting, cinematic slow zoom, premium tech commercial" # @param {"type":"string"}
negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走" # @param {"type":"string"}
width = 832 # @param {"type":"number"}
height = 480 # @param {"type":"number"}
seed = 82628696717258 # @param {"type":"integer"}
steps = 20 # @param {"type":"integer", "min":1, "max":100}
cfg_scale = 3 # @param {"type":"number", "min":1, "max":20}
sampler_name = "uni_pc" # @param ["uni_pc", "euler", "dpmpp_2m", "ddim", "lms"]
scheduler = "simple" # @param ["simple", "normal", "karras", "exponential"]
frames = 24 # @param {"type":"integer", "min":1, "max":120}   # 24 إطار ~ 1.5 ثانية (موثوق على T4) — زد لـ33 للمزيد من الثواني
fps = 16 # @param {"type":"integer", "min":1, "max":60}
output_format = "mp4" # @param ["mp4", "webm"]

output_path = generate_video(
    positive_prompt=positive_prompt,
    negative_prompt=negative_prompt,
    width=width,
    height=height,
    seed=seed,
    steps=steps,
    cfg_scale=cfg_scale,
    sampler_name=sampler_name,
    scheduler=scheduler,
    frames=frames,
    fps=fps,
    output_format=output_format
)

# 🎬 تنزيل المقطع على جهازك مباشرة
try:
    files.download(output_path)
except Exception as e:
    print("تعذّر التنزيل التلقائي:", e)
    print("ابحث عن ملفك في: /content/ComfyUI/output/ComfyUI.mp4")

## 📌 خلاصة وخطوات متابعة
- كل توليدة = لقطة **1.5–2 ثانية** (يمكنك رفع `frames` إلى 33 = ~2 ثانية، لكن أبطأ)
- ولّد **4–8 لقطات** بأفكار مختلفة → وصلها في CapCut (مجاناً) مع الشعار والنص → إنشاء إعلان كامل 15–30 ثانية
- **الترخيص:** Apache 2.0 — مقاطعك ملكك، و**استخدامها في إعلانات ممولة رسمياً مسموح** بلا علامة مائية
- **جلسة كولاب تنتهي؟** الملفات تُمحى → بعد كل توليدة انزّلها على جهازك (الخانة تفعل ذلك تلقائياً)
- **السرعة:** على كولاب المجاني T4 كل لقطة ~20–28 دقيقة — عادي؛ لتسريع اشتغل ليلاً أو اتركها تولّد عدة لقطات بجلسة واحدة

> 🔄 عند فتح جديد للجلسة: أعد تشغيل الخانة 1 (التحميل ~16GB ياخذ دقائق) ثم الخانة 2 — العملية مؤتمتة بالكامل.